# S7 Colour v8 — GPU development qualification

This notebook trains on the frozen **development** archive, selects architecture/epoch on validation only, and calibrates only the selected checkpoint. It never creates or reads the new external final and never enables SaaS integration.

## 1. Private Drive mount

Google will ask for authorization. Do not paste credentials or tokens into the notebook.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys

GITHUB_REPOSITORY = 'https://github.com/getibplay-cmyk/pfe.git'
GITHUB_BRANCH = 'codex/s7-color-v8-colab-gpu'
EXPECTED_ARCHIVE_SHA256 = 'ceb971e7af86194a56d0c33a4d10356c7174d911a33714d7e7336c27e164f62b'
DRIVE_PROJECT = Path('/content/drive/MyDrive/RentFleet_PFE/S7_vehicle_vision_assistant')
DRIVE_DATA = DRIVE_PROJECT / 'donnees_preparees' / 'S7_COLOR_V8'
DRIVE_MODELS = DRIVE_PROJECT / 'modeles_prives' / 'S7_COLOR_V8'
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
DRIVE_RUN = DRIVE_MODELS / RUN_ID
LOCAL_REPO = Path('/content/pfe')
LOCAL_DATA = Path('/content/s7_color_v8_development')
LOCAL_RUN = Path('/content/s7_color_v8_run')
DRIVE_RUN.mkdir(parents=True, exist_ok=False)
LOCAL_RUN.mkdir(parents=True, exist_ok=False)
print({'run_id': RUN_ID, 'drive_run': str(DRIVE_RUN)})

## 2. CUDA and immutable inputs

If this cell fails, choose **Runtime → Change runtime type → GPU**, then restart from the beginning. Colab GPU type and availability can vary.

In [ ]:
subprocess.run(['nvidia-smi'], check=True)
import torch
assert torch.cuda.is_available(), 'CUDA is mandatory for the real run'
print({'torch': torch.__version__, 'cuda': torch.version.cuda, 'gpu': torch.cuda.get_device_name(0)})

In [ ]:
subprocess.run(['git', 'clone', '--depth', '1', '--branch', GITHUB_BRANCH, GITHUB_REPOSITORY, str(LOCAL_REPO)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(LOCAL_REPO / 'scripts/intelligence/color_v8/requirements-color-v8.txt')], check=True)
git_sha = subprocess.check_output(['git', '-C', str(LOCAL_REPO), 'rev-parse', 'HEAD'], text=True).strip()
print({'git_sha': git_sha, 'branch': GITHUB_BRANCH})

In [ ]:
def sha256_file(path, chunk_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for chunk in iter(lambda: stream.read(chunk_size), b''):
            digest.update(chunk)
    return digest.hexdigest()

drive_archive = DRIVE_DATA / 'S7_COLOR_V8_DEVELOPMENT_DATA.zip'
drive_multipart = DRIVE_DATA / 'S7_COLOR_V8_DEVELOPMENT_DATA.zip.multipart.json'
drive_registry = DRIVE_DATA / 'S7_COLOR_V8_DEVELOPMENT_REGISTRY.json'
assert drive_registry.is_file(), f'Missing Drive registry in {DRIVE_DATA}'
registry = json.loads(drive_registry.read_text())
assert registry['status'] == 'DEVELOPMENT_ONLY_NEW_FINAL_NOT_CREATED'
assert registry['artifacts'][drive_archive.name]['sha256'] == EXPECTED_ARCHIVE_SHA256
local_archive = Path('/content/S7_COLOR_V8_DEVELOPMENT_DATA.zip')
if drive_archive.is_file():
    shutil.copy2(drive_archive, local_archive)
else:
    assert drive_multipart.is_file(), f'Missing archive and multipart registry in {DRIVE_DATA}'
    multipart = json.loads(drive_multipart.read_text())
    assert multipart['archive']['sha256'] == EXPECTED_ARCHIVE_SHA256
    with local_archive.open('wb') as destination:
        for expected_part in multipart['parts']:
            part_path = DRIVE_DATA / expected_part['name']
            assert part_path.is_file(), f'Missing multipart file: {part_path}'
            digest = hashlib.sha256(); copied = 0
            with part_path.open('rb') as source:
                for chunk in iter(lambda: source.read(8 * 1024 * 1024), b''):
                    destination.write(chunk); digest.update(chunk); copied += len(chunk)
            assert copied == expected_part['bytes'] and digest.hexdigest() == expected_part['sha256']
assert sha256_file(local_archive) == EXPECTED_ARCHIVE_SHA256
subprocess.run(['unzip', '-q', str(local_archive), '-d', str(LOCAL_DATA)], check=True)
manifest = LOCAL_DATA / 'S7_COLOR_V8_DEVELOPMENT_MANIFEST.csv'
assert sha256_file(manifest) == registry['artifacts'][manifest.name]['sha256']
print({'rows': registry['retained_rows'], 'manifest_sha256': sha256_file(manifest), 'final_status': registry['new_final']['status']})

## 3. Candidate training — validation only

Each completed candidate is checkpointed to Drive. The script structurally refuses calibration and final access.

In [ ]:
SCRIPT_DIR = LOCAL_REPO / 'scripts/intelligence/color_v8'
candidate_plan = [
    ('mobilenet_v3_large', 96),
    ('efficientnet_v2_s', 48),
    ('convnext_tiny', 64),
]
candidate_reports = []
for model_name, batch_size in candidate_plan:
    candidate_dir = LOCAL_RUN / 'candidates' / model_name
    command = [
        sys.executable, str(SCRIPT_DIR / 'train_color_v8.py'),
        '--dataset-root', str(LOCAL_DATA),
        '--manifest', str(manifest),
        '--output-dir', str(candidate_dir),
        '--model-name', model_name,
        '--epochs', '18', '--patience', '5',
        '--batch-size', str(batch_size), '--workers', '2',
    ]
    subprocess.run(command, check=True)
    report = next(candidate_dir.glob('*_CANDIDATE_REPORT.json'))
    candidate_reports.append(report)
    shutil.copytree(candidate_dir, DRIVE_RUN / 'candidates' / model_name)
    print({'checkpointed': model_name, 'drive': str(DRIVE_RUN / 'candidates' / model_name)})

## 4. Deterministic selection, then calibration of one candidate

In [ ]:
selection_dir = LOCAL_RUN / 'selection'
subprocess.run([
    sys.executable, str(SCRIPT_DIR / 'select_color_v8_candidate.py'),
    '--reports', *[str(path) for path in candidate_reports],
    '--output-dir', str(selection_dir),
], check=True)
shutil.copytree(selection_dir, DRIVE_RUN / 'selection')
selection_ledger = json.loads((selection_dir / 'S7_COLOR_V8_SELECTION_LEDGER.json').read_text())
print({'selected': selection_ledger['selected']['candidate'], 'calibration_loaded_during_selection': selection_ledger['calibration_images_loaded']})

In [ ]:
qualification_dir = LOCAL_RUN / 'qualification'
subprocess.run([
    sys.executable, str(SCRIPT_DIR / 'qualify_color_v8_development.py'),
    '--dataset-root', str(LOCAL_DATA),
    '--manifest', str(manifest),
    '--selection-dir', str(selection_dir),
    '--output-dir', str(qualification_dir),
    '--batch-size', '128', '--workers', '2',
], check=True)
shutil.copytree(qualification_dir, DRIVE_RUN / 'qualification')
qualification_report = json.loads((qualification_dir / 'S7_COLOR_V8_DEVELOPMENT_QUALIFICATION_REPORT.json').read_text())
run_ledger = {
    'schema_version': '8.0.0',
    'run_id': RUN_ID, 'git_sha': git_sha, 'github_branch': GITHUB_BRANCH,
    'gpu': torch.cuda.get_device_name(0),
    'development_archive_sha256': EXPECTED_ARCHIVE_SHA256,
    'selected_candidate': selection_ledger['selected']['candidate'],
    'development_gate_passed': qualification_report['decisions']['development_gate_passed'],
    'new_external_final_authorized': qualification_report['decisions']['new_external_final_authorized'],
    'new_external_final_executed': False,
    'saas_integration_authorized': False,
}
(DRIVE_RUN / 'S7_COLOR_V8_COLAB_RUN_LEDGER.json').write_text(json.dumps(run_ledger, indent=2, sort_keys=True) + '\n')
print(json.dumps(run_ledger, indent=2))

## STOP

This notebook intentionally stops here. Even when the development gate passes, do **not** improvise a final run. First freeze a newly sourced, prediction-blind, independently licensed final with `freeze_color_v8_external_final.py`; then execute `evaluate_color_v8_external_final_once.py` exactly once. ONNX export and SaaS integration remain forbidden until that report passes.